# Attention-selection evidence

In [ ]:
# Move to the project root and load notebook dependencies.
%cd /workspaces/tesis_v4

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
from torchvision import transforms

ROOT = Path.cwd()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.style.use('default')
print(f'Root: {ROOT} | Device: {DEVICE}')

## Load checkpoint and frame

In [ ]:
# Change CURRENT_MODEL only; older anchors remain available for deliberate comparisons.
MODEL_VARIANTS = {
    'current_beta0.1_seed42': ROOT / 'runs/component_refinement/02_vectorized_reward_anchor/beta0.1_seed42/checkpoint_best.pt',
    'current_beta0.1_seed43': ROOT / 'runs/component_refinement/02_vectorized_reward_anchor/beta0.1_seed43/checkpoint_best.pt',
    'corrected_beta0_seed42': ROOT / 'runs/component_refinement/01_corrected_reward_anchor/beta0.0_seed42/checkpoint_best.pt',
    'corrected_beta0.01_seed42': ROOT / 'runs/component_refinement/01_corrected_reward_anchor/beta0.01_seed42/checkpoint_best.pt',
    'corrected_beta0.1_seed42': ROOT / 'runs/component_refinement/01_corrected_reward_anchor/beta0.1_seed42/checkpoint_best.pt',
}
CURRENT_MODEL = 'current_beta0.1_seed42'
CHECKPOINT = MODEL_VARIANTS[CURRENT_MODEL]
ROLLOUT_DIR = ROOT / 'data/rollouts/rwm_deterministic/scenario_0'
FRAME_INDEX = 70  # Change deliberately and record interesting examples below.

In [ ]:
# Select one training rollout frame for visual debugging only.
from rwm.commands.rwm_manual_test import load_model

assert CHECKPOINT.exists(), f'Missing checkpoint: {CHECKPOINT}'
rollout_paths = sorted(ROLLOUT_DIR.rglob('*.npz'))
assert rollout_paths, f'No rollouts found: {ROLLOUT_DIR}'
ROLLOUT_PATH = rollout_paths[0]

with np.load(ROLLOUT_PATH) as rollout:
    raw_frames = rollout['obs']
assert 0 <= FRAME_INDEX < len(raw_frames), f'FRAME_INDEX must be below {len(raw_frames)}'

resize_to_model = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((64, 64), antialias=True),
])
frame_rgb = raw_frames[FRAME_INDEX]
frame_tensor = resize_to_model(Image.fromarray(frame_rgb)).unsqueeze(0).to(DEVICE)
model = load_model(CHECKPOINT, DEVICE)

print(f'Checkpoint: {CHECKPOINT.relative_to(ROOT)}')
print(f'Frame: {ROLLOUT_PATH.name} @ index {FRAME_INDEX} | source shape: {frame_rgb.shape}')

## Trace and render selection

In [ ]:
# Capture scorer logits, hard Top-K indices, and actual selected-patch pooling weights.
from rwm.evaluation.attention_trace import (
    trace_attention, render_heatmap, render_selected_overlay,
)

trace = trace_attention(model, frame_tensor)
heatmap = render_heatmap(trace)[0].numpy()
selected_overlay = render_selected_overlay(trace)[0].numpy()
model_frame = frame_tensor[0].detach().cpu().permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(model_frame)
axes[0].set_title('Model input (64×64)')
axes[1].imshow(model_frame)
axes[1].imshow(heatmap, cmap='magma', alpha=.58)
axes[1].set_title('All-patch scorer heatmap')
axes[2].imshow(model_frame)
axes[2].imshow(selected_overlay, cmap='Greens', alpha=.45, vmin=0, vmax=1)
axes[2].set_title(f'Hard Top-{trace.indices.shape[1]} selected patches')
for ax in axes:
    ax.axis('off')
fig.suptitle(f'{ROLLOUT_PATH.name} | frame {FRAME_INDEX}')
fig.tight_layout()
plt.show()

## Determinism and nearby-frame stability

In [ ]:
# Eval-mode Top-K must be deterministic for an identical frame.
repeat_trace = trace_attention(model, frame_tensor)
same_indices = torch.equal(trace.indices, repeat_trace.indices)
same_logits = torch.allclose(trace.logits, repeat_trace.logits)
print(f'Same indices: {same_indices} | Same logits: {same_logits}')
assert same_indices and same_logits, 'Eval-mode attention trace must be deterministic.'

In [ ]:
# Compare selected-patch overlap across nearby observations; this is descriptive, not a semantic claim.
OFFSETS = [-2, -1, 0, 1, 2]
nearby_rows = []
reference = set(selected.tolist())

for offset in OFFSETS:
    idx = FRAME_INDEX + offset
    if not 0 <= idx < len(raw_frames):
        continue
    image = resize_to_model(Image.fromarray(raw_frames[idx])).unsqueeze(0).to(DEVICE)
    nearby_trace = trace_attention(model, image)
    indices = set(nearby_trace.indices[0].numpy().tolist())
    union = reference | indices
    nearby_rows.append({
        'frame': idx, 'offset': offset, 'shared_top_k': len(reference & indices),
        'jaccard_overlap': len(reference & indices) / len(union),
    })

stability = pd.DataFrame(nearby_rows)
display(stability.style.format({'jaccard_overlap': '{:.3f}'}))

## Record an evidence image

In [ ]:
# Save the current three-panel figure only when it is useful evidence; do not commit exploratory images by default.
EVIDENCE_DIR = ROOT / 'runs/evidence/attention_selection'
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
output_path = EVIDENCE_DIR / f'{ROLLOUT_PATH.stem}_frame{FRAME_INDEX:04d}.png'
fig.savefig(output_path, dpi=160, bbox_inches='tight')
print(f'Saved: {output_path.relative_to(ROOT)}')